# EarlyAlert: 01 - Data Ingestion & Audit
**Author:** Vasanthkumar A | **Dept:** Computer Science and Business Systems | **Knowledge Institute of Technology**

This notebook demonstrates the exploratory data audit and schema validation for the UCI Student Dropout and Academic Success dataset (ID 697).

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root in sys.path
sys.path.insert(0, os.path.abspath('..'))

from src.validate import validate_schema, CATEGORICAL_CODE_COLUMNS, CONTINUOUS_NUMERIC_COLUMNS

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load Raw Dataset & Validate Schema (FR-01)

In [ ]:
raw_path = os.path.join('..', 'data', 'raw', 'student_dropout.csv')
df = pd.read_csv(raw_path)
print(f'Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')

# Run strict schema validation
is_valid = validate_schema(df, is_inference=False)
print(f'Schema Validation Passed: {is_valid}')

## 2. Target Variable Class Distribution

In [ ]:
target_counts = df['Target'].value_counts()
target_pct = df['Target'].value_counts(normalize=True) * 100

summary_target = pd.DataFrame({'Count': target_counts, 'Percentage (%)': target_pct.round(2)})
print(summary_target)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=target_counts.index, y=target_counts.values, palette='Blues_r', ax=ax)
ax.set_title('Target Distribution: Student Academic Outcome', fontsize=12, fontweight='bold')
ax.set_xlabel('Outcome Category')
ax.set_ylabel('Number of Students')
for i, v in enumerate(target_counts.values):
    ax.text(i, v + 30, f'{v} ({target_pct.iloc[i]:.1f}%)', ha='center', fontweight='bold')
plt.show()

## 3. Data Audit Table (FR-02)

In [ ]:
audit_table_path = os.path.join('..', 'reports', 'audit_table.csv')
audit_df = pd.read_csv(audit_table_path)
print(f'Total Columns Audited: {len(audit_df)}')
print(audit_df[['Column Name', 'Feature Category', 'Data Type', 'Missing %', 'Unique Values']].head(15))

## 4. Key Predictors vs Dropout Risk

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Tuition fees up to date vs Target
tuition_cross = pd.crosstab(df['Tuition fees up to date'], df['Target'], normalize='index') * 100
tuition_cross.plot(kind='bar', stacked=True, ax=axes[0], color=['#e74c3c', '#f39c12', '#2ecc71'])
axes[0].set_title('Tuition Fees Status vs Academic Outcome', fontweight='bold')
axes[0].set_xticklabels(['Overdue (0)', 'Up to Date (1)'], rotation=0)
axes[0].set_ylabel('Percentage (%)')

# 2. Approved credits 1st sem vs Target
sns.boxplot(x='Target', y='Curricular units 1st sem (approved)', data=df, palette='Set2', ax=axes[1])
axes[1].set_title('1st Sem Approved Units by Academic Outcome', fontweight='bold')

plt.tight_layout()
plt.show()